# Many Models Forecasting Demo

This notebook showcases how to run MMF with MLForecast-based global models on multiple time series of daily resolution **using exogenous regressors**. We use [`MLForecastLGBM`](https://github.com/databricks-industry-solutions/many-model-forecasting/blob/main/mmf_sa/models/mlforecast/MLForecastPipeline.py) (fixed) and [`MLForecastAutoLGBM`](https://github.com/databricks-industry-solutions/many-model-forecasting/blob/main/mmf_sa/models/mlforecast/MLForecastPipeline.py) (Optuna-tuned) on the [Rossmann Store Sales](https://www.kaggle.com/competitions/rossmann-store-sales/data) dataset. To run this notebook you need to download the Rossmann dataset from Kaggle and stage it in a Unity Catalog volume.

Unlike the [`global_external_regressors_daily_dl.ipynb`](https://github.com/databricks-industry-solutions/many-model-forecasting/blob/main/examples/external_regressors/global_external_regressors_daily_dl.ipynb) notebook (GPU + NeuralForecast), this notebook stays on CPU and runs end-to-end in a single Python process.

### Cluster setup

We recommend using a **single-node CPU cluster** with [Databricks Runtime 17.3 LTS for ML](https://docs.databricks.com/en/release-notes/runtime/17.3lts-ml.html). The pinned versions in [`requirements-global.txt`](https://github.com/databricks-industry-solutions/many-model-forecasting/blob/main/requirements-global.txt) match the DBR 17.3 LTS ML preinstalled versions exactly.

### Install and import packages

In [ ]:
%pip install -r ../../requirements-global.txt --quiet
dbutils.library.restartPython()

In [ ]:
import logging
import random
from mmf_sa import run_forecast

logging.getLogger("py4j.clientserver").setLevel(logging.WARNING)
logging.getLogger("py4j.java_gateway").setLevel(logging.WARNING)

### Prepare data
Before running this notebook, download the Rossmann dataset from [Kaggle](https://www.kaggle.com/competitions/rossmann-store-sales/data) and stage `train.csv` + `test.csv` in a Unity Catalog [volume](https://docs.databricks.com/en/connect/unity-catalog/volumes.html).

In [ ]:
catalog = "mmf"  # Name of the catalog we use to manage our assets
db = "rossmann"  # Name of the schema we use to manage our assets
volume = "csv"   # Name of the volume where you have your rossmann csv files
user = spark.sql('select current_user() as user').collect()[0]['user']  # User email address

In [ ]:
_ = spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalog}")
_ = spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{db}")
_ = spark.sql(f"CREATE VOLUME IF NOT EXISTS {catalog}.{db}.{volume}")

In [ ]:
# Randomly select 100 stores to forecast
random.seed(7)
sample = True
size = 100
stores = sorted(random.sample(range(0, 1000), size))

train = spark.read.csv(f"/Volumes/{catalog}/{db}/{volume}/train.csv", header=True, inferSchema=True)
test = spark.read.csv(f"/Volumes/{catalog}/{db}/{volume}/test.csv", header=True, inferSchema=True)

if sample:
    train = train.filter(train.Store.isin(stores))
    test = test.filter(test.Store.isin(stores))

In [ ]:
train.write.mode("overwrite").option("mergeSchema", "true").saveAsTable(f"{catalog}.{db}.rossmann_daily_train")
test.write.mode("overwrite").option("mergeSchema", "true").saveAsTable(f"{catalog}.{db}.rossmann_daily_test")

In [ ]:
display(spark.sql(f"select * from {catalog}.{db}.rossmann_daily_train where Store=49 order by Date limit 10"))
display(spark.sql(f"select * from {catalog}.{db}.rossmann_daily_test where Store=49 order by Date limit 10"))

Note that `rossmann_daily_train` carries the target `Sales` whereas `rossmann_daily_test` does not — the test table supplies `dynamic_future_categorical` regressor values for future dates. MMF's scoring path unions train and test so the model can produce forecasts conditioned on the future regressor values.

### Models

We use four integer-valued future-known regressors that exist in both `train` and `test`: `DayOfWeek`, `Open`, `Promo`, `SchoolHoliday`. We exclude `StateHoliday` (string-typed; LightGBM does not consume strings without explicit categorical encoding). The model adapter ([`MLForecastPipeline.py`](https://github.com/databricks-industry-solutions/many-model-forecasting/blob/main/mmf_sa/models/mlforecast/MLForecastPipeline.py)) constructs a complete future-regressor `X_df` per backtest window via the `new_df` mechanism.

In [ ]:
active_models = [
    "MLForecastLGBM",
    "MLForecastAutoLGBM",
]

### Run MMF

In [ ]:
run_forecast(
    spark=spark,
    train_data=f"{catalog}.{db}.rossmann_daily_train",
    scoring_data=f"{catalog}.{db}.rossmann_daily_test",
    scoring_output=f"{catalog}.{db}.rossmann_daily_scoring_output",
    evaluation_output=f"{catalog}.{db}.rossmann_daily_evaluation_output",
    model_output=f"{catalog}.{db}",
    group_id="Store",
    date_col="Date",
    target="Sales",
    freq="D",
    prediction_length=14,
    backtest_length=28,
    stride=14,
    metric="smape",
    dynamic_future_categorical=["DayOfWeek", "Open", "Promo", "SchoolHoliday"],
    train_predict_ratio=1,
    data_quality_check=True,
    resample=False,
    active_models=active_models,
    experiment_path=f"/Users/{user}/mmf/rossmann_daily_ml",
    use_case_name="rossmann_daily",
    accelerator="cpu",
)

### Evaluate

In [ ]:
display(
  spark.sql(f"""
    select * from {catalog}.{db}.rossmann_daily_evaluation_output
    where Store=49 and model in ('MLForecastLGBM', 'MLForecastAutoLGBM')
    order by Store, model, backtest_window_start_date
  """)
)

### Forecast

In [ ]:
display(spark.sql(f"""
    select * from {catalog}.{db}.rossmann_daily_scoring_output
    where Store=49 and model in ('MLForecastLGBM', 'MLForecastAutoLGBM')
    order by Store, model
    """))

### Delete Tables

In [ ]:
#display(spark.sql(f"delete from {catalog}.{db}.rossmann_daily_evaluation_output where model in ('MLForecastLGBM', 'MLForecastAutoLGBM')"))

In [ ]:
#display(spark.sql(f"delete from {catalog}.{db}.rossmann_daily_scoring_output where model in ('MLForecastLGBM', 'MLForecastAutoLGBM')"))